In [4]:
import sys
!{sys.executable} -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 5.4 MB/s  0:00:02 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [5]:
import spacy
from spacy.pipeline import EntityRuler

# Load the base English model
nlp = spacy.load("en_core_web_sm")

print(f"Spacy Pipeline components: {nlp.pipe_names}")

Spacy Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [6]:
# Our dictionary of skills (The "Lexicon")
# We lowercase everything because we already lowercased our resume text in Phase 2.
tech_skills = [
    # Languages
    "python", "java", "c++", "c#", "javascript", "typescript", "golang", "rust", "php", "ruby", "sql", "html", "css",
    # Frameworks/Libraries
    "react", "angular", "vue", "django", "fastapi", "flask", "spring boot", "node.js", "express", "pandas", "numpy",
    "scikit-learn", "tensorflow", "keras", "pytorch", "opencv", "matplotlib", "seaborn",
    # Databases
    "postgresql", "mysql", "mongodb", "redis", "cassandra", "oracle", "elasticsearch",
    # Cloud & DevOps
    "aws", "azure", "gcp", "docker", "kubernetes", "jenkins", "github actions", "gitlab ci", "terraform", "ansible",
    "linux", "bash", "git",
    # Concepts
    "machine learning", "deep learning", "natural language processing", "nlp", "computer vision", "data analysis",
    "agile", "scrum", "rest api", "graphql", "microservices", "ci/cd"
]

print(f"Loaded {len(tech_skills)} custom skills into the lexicon.")

Loaded 63 custom skills into the lexicon.


In [7]:
# 1. Create a blank EntityRuler and add it to the pipeline BEFORE the 'ner' component
# We put it before 'ner' so our rules override Spacy's default guesses.
if "entity_ruler" not in nlp.pipe_names:
    ruler = nlp.add_pipe("entity_ruler", before="ner")
else:
    ruler = nlp.get_pipe("entity_ruler")

# 2. Format our lexicon into the exact dictionary format Spacy requires
# Example: {"label": "SKILL", "pattern": "python"}
patterns = [{"label": "SKILL", "pattern": skill} for skill in tech_skills]

# 3. Add the patterns to the ruler
ruler.add_patterns(patterns)

print(f"Updated Pipeline components: {nlp.pipe_names}")

Updated Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'entity_ruler', 'ner']


In [8]:
def extract_skills(text):
    """Passes text through Spacy and extracts recognized skills."""
    # Process the text
    doc = nlp(text)
    
    extracted_skills = []
    
    # Iterate through every recognized entity
    for ent in doc.ents:
        if ent.label_ == "SKILL":
            extracted_skills.append(ent.text)
            
    # Convert to a set to remove duplicates, then back to a sorted list
    return sorted(list(set(extracted_skills)))

In [9]:
# A fake, cleaned resume snippet
test_text = """
senior software engineer with 5 years of experience building scalable microservices. 
proficient in python, javascript, and sql. deployed applications using docker and kubernetes on aws. 
strong background in machine learning using tensorflow and scikit-learn. 
familiar with agile methodologies and ci/cd pipelines using github actions.
"""

print("Processing text...")
found_skills = extract_skills(test_text)

print("\n--- SKILLS FOUND ---")
for skill in found_skills:
    print(f"- {skill.title()}")
    
print(f"\nTotal skills extracted: {len(found_skills)}")

Processing text...

--- SKILLS FOUND ---
- Agile
- Aws
- Ci/Cd
- Docker
- Github Actions
- Javascript
- Kubernetes
- Machine Learning
- Microservices
- Python
- Scikit-Learn
- Sql
- Tensorflow

Total skills extracted: 13
